# 08a — Tools and bounded workflows — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Python standard library only; Jupyter is needed for the notebook UI.

[Stage guide](../stages/08.md) · [Course map](../PLAN.md) · [Project](../../projects/stage08/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Validate tool names and arguments
- Separate decisions from execution
- Bound retries, steps, and side effects


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Tools and bounded workflows

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A tool is a named callable operation with a defined input/output schema. A workflow controls which tools execute and in what order. A model may propose arguments, but trusted code validates the schema, permissions, and allowed tool name before execution.

State records progress; a trace records observations and failures. A step budget limits the number of actions. A deadline bounds overall elapsed time; each network/database operation also needs its own timeout because a between-step check cannot interrupt a blocked call.

Retries are appropriate only for defined transient failures. Idempotency means repeating the same request does not duplicate its effect. A read often permits retry; a payment-like action needs a durable deduplication protocol.

### Worked example: follow the values

Plan: search for a statement, read a scoped financial record, divide two verified values. With max_calls=2, stop before the third call and report budget exhaustion. If the first tool receives an extra tenant argument from model text, reject it instead of letting it override the authenticated tenant.

### Pause and explain

Why return an explicit failed observation instead of inventing a tool result?

<details><summary>Check your reasoning after trying</summary>

Downstream decisions must know that evidence is unavailable; invented results hide the failure and can trigger incorrect actions.

</details>

### Common mistakes to check

A schema validates shape, not authority. Approval must be bound to the exact action, actor, expiry, and single-use status.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

A tool call is a structured request to execute a function. A model proposes a name and arguments; application code validates them, authorizes the operation, and executes it. Tool output is an observation, not a new instruction with higher authority. Never invent a result when a tool fails.

A workflow has state, transitions, and terminal outcomes. A fixed sequence is often sufficient. An agent adds model-selected transitions, which increases the need for budgets and evaluation. Bound calls, elapsed time, retries, response sizes, and concurrency. A deadline checked between calls does not cancel a blocking call; each I/O operation needs its own timeout.

Read-only tools can often be retried. Side effects need idempotency keys and explicit approval tied to the exact action, arguments, actor, and expiration. Persist only necessary state, separating users and runs. Multiple agents add coordination and failure paths; use them only if a measured comparison justifies the cost.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
request = {"tool": "divide", "arguments": {"a": 12, "b": 3}}
print(request)


## Exercise 1: Allowlisted calculator

Execute add or divide from a request with exactly tool,arguments and arguments exactly a,b; finite numeric values only (exclude booleans). Reject all other requests.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def execute_tool(request):
    import math
    if set(request) != {"tool", "arguments"}: raise ValueError("Invalid request")
    args = request["arguments"]
    if set(args) != {"a", "b"}: raise ValueError("Invalid arguments")
    if any(type(v) not in (int, float) or not math.isfinite(v) for v in args.values()): raise ValueError("Invalid number")
    a, b = args["a"], args["b"]
    if request["tool"] == "add": return a+b
    if request["tool"] == "divide": return a/b
    raise ValueError("Unknown tool")


In [ ]:
assert execute_tool({"tool": "divide", "arguments": {"a": 12, "b": 3}}) == 4
expect_error(ValueError, lambda: execute_tool({"tool": "shell", "arguments": {"a": 1, "b": 2}}))
print("Exercise 1: checks passed")


**Why this works:** A small typed allowlist prevents model text from becoming arbitrary code.


## Exercise 2: Call budget

Execute sequential requests up to max_calls. Return observations and status complete or budget_exhausted. Record tool errors as strings; do not pretend they succeeded.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def run_tools(requests, max_calls=3):
    observations = []
    for request in requests[:max_calls]:
        try: observations.append({"result": execute_tool(request)})
        except (ValueError, ZeroDivisionError) as exc: observations.append({"error": type(exc).__name__})
    return observations, "budget_exhausted" if len(requests) > max_calls else "complete"


In [ ]:
request = {"tool": "add", "arguments": {"a": 1, "b": 2}}
observations, status = run_tools([request]*5, 2)
assert len(observations) == 2 and status == "budget_exhausted"
print("Exercise 2: checks passed")


**Why this works:** A model instruction to stop is weaker than a code-enforced limit.


## Exercise 3: Retry only transient errors

Call operation at most attempts times; retry TimeoutError only. Raise the final error if exhausted.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def retry(operation, attempts=3):
    if attempts < 1: raise ValueError("Positive attempts required")
    for i in range(attempts):
        try: return operation()
        except TimeoutError:
            if i == attempts-1: raise


In [ ]:
calls = []
def flaky():
    calls.append(1)
    if len(calls) < 2: raise TimeoutError()
    return "ok"
assert retry(flaky) == "ok" and len(calls) == 2
expect_error(ValueError, lambda: retry(lambda: int("bad")))
print("Exercise 3: checks passed")


**Why this works:** Real network retries also need backoff, jitter, and a total deadline.


## Independent transfer

Extend document search with typed SQL read tools and a calculator. Add traces, step/deadline budgets, timeouts, approval records, and tests for malformed tool arguments and unavailable services.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [08a interface and acceptance cases](../assignments/08a.md).
2. Copy the [blank starter](../assignments/starters/08a.py) to `work/assignments/08a.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 08a --solution work/assignments/08a.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/08a.md#08a-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 08a --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. Who grants tool permission?

   Application authorization policy using verified identity, never the model’s claim about its own authority.

2. When is retry unsafe?

   When a side effect may have happened and the request is not idempotent.


## Reading and review

- [Primary reference 1](https://docs.langchain.com/oss/python/langgraph/overview)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
